# 🎬 Movie Recommendation System — Pattern Mining
### Member 2 — Association Rule Mining Specialist

**Goal:** Discover frequent movie combinations and generate association rules like *"Users who watched X also watched Y"*

**Input:** `transactions.csv` from Member 1  
**Output:** `frequent_itemsets.csv` + `association_rules.csv`

---
##  Step 1 — Mount Google Drive & Install Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install mlxtend -q

---
##  Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

print(' All libraries imported successfully!')

---
##  Step 3 — Set File Paths

>  **Change `BASE` to match your Google Drive folder path**

In [ ]:
# =============================================
#  CHANGE THIS TO YOUR DRIVE FOLDER PATH
BASE = '/content/drive/MyDrive/DataMining_Project'
# =============================================

TRANSACTIONS_PATH = f'{BASE}/data/processed/transactions.csv'
OUTPUT_DIR        = f'{BASE}/outputs'
OUTPUT_ITEMSETS   = f'{OUTPUT_DIR}/frequent_itemsets.csv'
OUTPUT_RULES      = f'{OUTPUT_DIR}/association_rules.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(' Paths configured!')
print(f'   Transactions: {TRANSACTIONS_PATH}')
print(f'   Outputs:      {OUTPUT_DIR}')

---
##  Step 4 — Load & Understand the Data

In [ ]:
trans_df = pd.read_csv(TRANSACTIONS_PATH)

print('Shape:', trans_df.shape)
print('Columns:', trans_df.columns.tolist())
print()
trans_df.head()

In [ ]:
# Check what the 'title' column looks like (string or real list?)
print('Type of first value:', type(trans_df['title'][0]))
print('Sample value:', trans_df['title'][0][:100])

---
##  Step 5 — Convert Title Column to Real Python Lists

In [ ]:
# The title column was saved as a string like "['movie1', 'movie2']"
# We need to convert it back to a real Python list
trans_df['title'] = trans_df['title'].apply(ast.literal_eval)

# Verify conversion
print('Type after conversion:', type(trans_df['title'][0]))
print('Sample (first 5 movies of user 1):', trans_df['title'][0][:5])
print()
print(f'Total users (transactions): {len(trans_df)}')
print(f'Avg movies per user: {trans_df["title"].apply(len).mean():.1f}')

---
## Step 6 — One-Hot Encoding (TransactionEncoder)

Convert the list of movies per user into a **True/False matrix** that the algorithm can process.

In [ ]:
transactions_list = trans_df['title'].tolist()

te = TransactionEncoder()
te_array = te.fit_transform(transactions_list)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f' Encoded Matrix Shape: {df_encoded.shape}')
print(f'   → {df_encoded.shape[0]} users  x  {df_encoded.shape[1]} unique movies')
df_encoded.head(3)

---
##  Step 7 — Apply FP-Growth Algorithm

`min_support = 0.05` means: only keep movie combinations watched by **at least 5%** of users.

In [ ]:
frequent_itemsets = fpgrowth(
    df_encoded,
    min_support=0.05,
    use_colnames=True
)

# Add length column (how many movies in the combination)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f' Found {len(frequent_itemsets)} frequent itemsets')
print()
print('Breakdown by combination size:')
print(frequent_itemsets['length'].value_counts().sort_index())
print()
frequent_itemsets.head(20)

---
##  Step 8 — Generate Association Rules

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1.0
)

rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f' Generated {len(rules)} association rules')
print()
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

---
##  Step 9 — Filter Strong Rules Only

Keep only rules where **Confidence ≥ 0.5** and **Lift ≥ 1.5**

In [ ]:
strong_rules = rules[
    (rules['confidence'] >= 0.5) &
    (rules['lift'] >= 1.5)
].copy().reset_index(drop=True)

print(f'Total rules:        {len(rules)}')
print(f'Strong rules only:  {len(strong_rules)}')
print()
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)

---
##  Step 10 — Generate Human-Readable Insights

In [ ]:
print('=' * 65)
print('  TOP MOVIE RECOMMENDATION INSIGHTS')
print('=' * 65)

for i, row in strong_rules.head(10).iterrows():
    ante = list(row['antecedents'])
    cons = list(row['consequents'])
    print(f'\n#{i+1}')
    print(f'  If user watched : {ante}')
    print(f'  They likely watched: {cons}')
    print(f'  Support:    {row["support"]:.2%}')
    print(f'  Confidence: {row["confidence"]:.2%}')
    print(f'  Lift:       {row["lift"]:.2f}x')

---
##  Step 11 — Visualizations

In [ ]:
# --- Plot 1: Top 15 Most Frequent Single Movies ---
single_items = frequent_itemsets[frequent_itemsets['length'] == 1].head(15).copy()
single_items['movie'] = single_items['itemsets'].apply(lambda x: list(x)[0])

plt.figure(figsize=(13, 5))
bars = plt.bar(single_items['movie'], single_items['support'], color='steelblue', edgecolor='white')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.title('Top 15 Most Watched Movies (by Support)', fontsize=13, fontweight='bold')
plt.xlabel('Movie')
plt.ylabel('Support')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/top_movies_support.png', dpi=150)
plt.show()
print(' Saved: top_movies_support.png')

In [ ]:
# --- Plot 2: Confidence Distribution ---
plt.figure(figsize=(8, 4))
plt.hist(rules['confidence'], bins=25, color='coral', edgecolor='white')
plt.title('Distribution of Rule Confidence', fontsize=13, fontweight='bold')
plt.xlabel('Confidence')
plt.ylabel('Number of Rules')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confidence_distribution.png', dpi=150)
plt.show()
print(' Saved: confidence_distribution.png')

In [ ]:
# --- Plot 3: Confidence vs Lift Scatter ---
plt.figure(figsize=(8, 5))
plt.scatter(rules['confidence'], rules['lift'], alpha=0.4, color='mediumseagreen', s=20)
plt.axhline(y=1.5, color='red', linestyle='--', linewidth=1, label='Lift = 1.5 threshold')
plt.axvline(x=0.5, color='blue', linestyle='--', linewidth=1, label='Confidence = 0.5 threshold')
plt.title('Confidence vs Lift (all rules)', fontsize=13, fontweight='bold')
plt.xlabel('Confidence')
plt.ylabel('Lift')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confidence_vs_lift.png', dpi=150)
plt.show()
print(' Saved: confidence_vs_lift.png')

In [ ]:
# --- Plot 4: Top 10 Rules by Lift (bar chart) ---
top10 = strong_rules.head(10).copy()
top10['rule_label'] = top10.apply(
    lambda r: f"{list(r['antecedents'])[0][:20]}... → {list(r['consequents'])[0][:20]}...", axis=1
)

plt.figure(figsize=(12, 5))
plt.barh(top10['rule_label'][::-1], top10['lift'][::-1], color='mediumpurple', edgecolor='white')
plt.title('Top 10 Strongest Rules (by Lift)', fontsize=13, fontweight='bold')
plt.xlabel('Lift')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/top10_rules_lift.png', dpi=150)
plt.show()
print(' Saved: top10_rules_lift.png')

---
##  Step 12 — Save Outputs to Drive

In [ ]:
# Convert frozensets to lists before saving (CSV can't store frozensets)
fi_to_save = frequent_itemsets.copy()
fi_to_save['itemsets'] = fi_to_save['itemsets'].apply(lambda x: list(x))
fi_to_save.to_csv(OUTPUT_ITEMSETS, index=False)

rules_to_save = rules.copy()
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda x: list(x))
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda x: list(x))
rules_to_save.to_csv(OUTPUT_RULES, index=False)

print(' All outputs saved successfully!')
print(f'    frequent_itemsets.csv  →  {OUTPUT_ITEMSETS}')
print(f'    association_rules.csv  →  {OUTPUT_RULES}')

---
##  Step 13 — Final Summary

In [ ]:
print('=' * 55)
print('           PATTERN MINING — FINAL SUMMARY')
print('=' * 55)
print(f'  Total users analyzed:         {len(trans_df)}')
print(f'  Unique movies:                {df_encoded.shape[1]}')
print(f'  Frequent itemsets found:      {len(frequent_itemsets)}')
print(f'  Total association rules:      {len(rules)}')
print(f'  Strong rules (conf≥0.5, lift≥1.5): {len(strong_rules)}')
print()
print('   Output files:')
print(f'     - frequent_itemsets.csv')
print(f'     - association_rules.csv')
print(f'     - top_movies_support.png')
print(f'     - confidence_distribution.png')
print(f'     - confidence_vs_lift.png')
print(f'     - top10_rules_lift.png')
print('=' * 55)